# DeFi Data Pipeline — FinTech 590
Fetches top Uniswap V3 pools, verifies contracts on Sourcify, and saves results.

## 0. Install Dependencies

In [1]:
import subprocess, sys

def ensure(*packages):
    for pkg in packages:
        try:
            __import__(pkg.replace("-", "_"))
        except ImportError:
            print(f"[setup] Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

ensure("requests", "pandas", "python-dotenv", "web3", "pyarrow")
print("All dependencies ready.")

[setup] Installing python-dotenv...
All dependencies ready.


## 1. Config

In [ ]:
import os, json, time, pathlib
import requests
import pandas as pd
from dotenv import load_dotenv
from web3 import Web3

load_dotenv()

DEFILLAMA_POOLS  = "https://yields.llama.fi/pools"
SOURCIFY_CHECK   = "https://sourcify.dev/server/v2/contract/{chain_id}/{address}"
SOURCIFY_FILES   = "https://sourcify.dev/server/files/any/{chain_id}/{address}"
RATE_LIMIT_DELAY = 0.25
MIN_TVL_USD      = 1_000_000
TOP_N            = 20

CHAINS = ["Ethereum", "Arbitrum", "Optimism", "Base", "Polygon"]

CHAIN_IDS = {
    "Ethereum": 1,
    "Arbitrum": 42161,
    "Optimism": 10,
    "Base":     8453,
    "Polygon":  137,
}

BLOCKSCOUT_URLS = {
    "Ethereum": "https://eth.blockscout.com",
    "Arbitrum": "https://arbitrum.blockscout.com",
    "Optimism": "https://optimism.blockscout.com",
    "Base":     "https://base.blockscout.com",
    "Polygon":  "https://polygon.blockscout.com",
}

# Uniswap V3 factory constants (Ethereum mainnet)
UNISWAP_V3_FACTORY  = "0x1F98431c8aD98523631AE4a59f267346ea31F984"
POOL_INIT_CODE_HASH = "e34f199b19b2b4f47f68442619d555527d244f78a3297ea89325f843f87b8b54"

ABI_DIR      = pathlib.Path("pool_abis")
PARQUET_PATH = pathlib.Path("data/top_pools.parquet")
ABI_DIR.mkdir(exist_ok=True)
PARQUET_PATH.parent.mkdir(exist_ok=True)

print(f"Output: {PARQUET_PATH}  |  ABIs: {ABI_DIR}/")
print(f"Chains : {', '.join(CHAINS)}")
print("Contract verification: Sourcify → Blockscout fallback (no API key required)")

## 2. Fetch Top Uniswap V3 Pools from DeFiLlama

> DeFiLlama returns internal UUIDs rather than on-chain addresses, so we compute the real
> pool address from token addresses + fee tier using Uniswap V3's deterministic CREATE2 formula.

In [3]:
def compute_pool_address(token_a: str, token_b: str, fee: int) -> str:
    """
    Derive the Uniswap V3 pool address from its two tokens and fee tier
    using the same CREATE2 formula as the factory contract.
    """
    token0, token1 = sorted([token_a.lower(), token_b.lower()])
    encoded = (
        bytes.fromhex(token0[2:]).rjust(32, b'\x00') +
        bytes.fromhex(token1[2:]).rjust(32, b'\x00') +
        fee.to_bytes(32, 'big')
    )
    salt = Web3.keccak(encoded)
    data = (
        b'\xff' +
        bytes.fromhex(UNISWAP_V3_FACTORY[2:]) +
        salt +
        bytes.fromhex(POOL_INIT_CODE_HASH)
    )
    return Web3.to_checksum_address('0x' + Web3.keccak(data)[12:].hex())


def parse_fee_tier(pool_meta: str | None) -> int:
    """Convert poolMeta string like '0.05%' to Uniswap fee pips (500)."""
    try:
        return int(float(pool_meta.replace("%", "")) * 10_000)
    except (AttributeError, ValueError):
        return 0


print("Fetching pool data from DeFiLlama...")
resp = requests.get(DEFILLAMA_POOLS, timeout=30)
resp.raise_for_status()
all_pools = resp.json()["data"]
print(f"  Total pools across all protocols: {len(all_pools):,}")

pools = []
for chain in CHAINS:
    chain_pools = [
        p for p in all_pools
        if p.get("project") == "uniswap-v3"
        and p.get("chain") == chain
        and (p.get("tvlUsd") or 0) >= MIN_TVL_USD
    ]
    chain_pools.sort(key=lambda x: x.get("tvlUsd", 0), reverse=True)
    top_raw = chain_pools[:TOP_N]
    print(f"  {chain}: {len(chain_pools)} qualifying pools, keeping top {len(top_raw)}")

    for p in top_raw:
        underlying   = p.get("underlyingTokens") or []
        fee          = parse_fee_tier(p.get("poolMeta"))
        symbol_parts = p.get("symbol", "-").split("-")
        address = (
            compute_pool_address(underlying[0], underlying[1], fee)
            if len(underlying) >= 2 and fee > 0
            else p["pool"]
        )
        pools.append({
            "address":    address,
            "chain":      chain,
            "token0":     symbol_parts[0] if len(symbol_parts) > 0 else "",
            "token1":     symbol_parts[1] if len(symbol_parts) > 1 else "",
            "fee_tier":   fee,
            "tvl_usd":    float(p.get("tvlUsd") or 0),
            "volume_usd": float(p.get("volumeUsd1d") or 0),
            "llama_id":   p["pool"],
        })

print(f"\n  Total pools selected: {len(pools)}")
print()
for i, p in enumerate(pools, 1):
    print(f"  {i:>3}. [{p['chain']:<9}] {p['token0']}/{p['token1']} "
          f"(fee {p['fee_tier']/1e4:.2f}%)  "
          f"TVL=${p['tvl_usd']:>14,.0f}  "
          f"addr={p['address'][:10]}...")

Fetching pool data from DeFiLlama...
  Total pools across all protocols: 18,544
  Ethereum: 136 qualifying pools, keeping top 20
  Arbitrum: 18 qualifying pools, keeping top 18
  Optimism: 1 qualifying pools, keeping top 1
  Base: 21 qualifying pools, keeping top 20
  Polygon: 4 qualifying pools, keeping top 4

  Total pools selected: 63

    1. [Ethereum ] USDC/WETH (fee 0.05%)  TVL=$    95,980,263  addr=0x88e6A0c2...
    2. [Ethereum ] WETH/USDT (fee 0.30%)  TVL=$    64,839,313  addr=0x4e68Ccd3...
    3. [Ethereum ] WBTC/WETH (fee 0.05%)  TVL=$    46,502,714  addr=0x4585FE77...
    4. [Ethereum ] WBTC/WETH (fee 0.30%)  TVL=$    44,304,648  addr=0xCBCdF962...
    5. [Ethereum ] WBTC/CBBTC (fee 0.01%)  TVL=$    29,150,519  addr=0xe8f7c89C...
    6. [Ethereum ] WHITE/WETH (fee 0.01%)  TVL=$    28,821,407  addr=0xC5c134A1...
    7. [Ethereum ] WBTC/USDC (fee 0.30%)  TVL=$    27,601,914  addr=0x99ac8cA7...
    8. [Ethereum ] USDC/USDT (fee 0.01%)  TVL=$    26,885,479  addr=0x3416cF6C...
 

## 3. Verify Each Pool Contract on Sourcify + Blockscout

> Tries [Sourcify](https://sourcify.dev) first (decentralized, no API key, chain-aware).  
> Falls back to [Blockscout](https://blockscout.com) if not found on Sourcify (covers all 5 chains).

In [ ]:
def sourcify_check(address: str, chain_id: int) -> tuple[bool, list | None]:
    """Check Sourcify for verification and ABI. Returns (is_verified, abi_or_None)."""
    try:
        r = requests.get(SOURCIFY_CHECK.format(chain_id=chain_id, address=address), timeout=15)
        if r.status_code == 404:
            return False, None
        r.raise_for_status()
        if not r.json().get("match"):
            return False, None

        r2 = requests.get(SOURCIFY_FILES.format(chain_id=chain_id, address=address), timeout=15)
        r2.raise_for_status()
        files = r2.json().get("files", [])
        meta  = next((f for f in files if f["name"] == "metadata.json"), None)
        if meta:
            abi = json.loads(meta["content"])["output"]["abi"]
            return True, abi
        return True, None

    except requests.RequestException:
        return False, None


def blockscout_check(address: str, chain: str) -> tuple[bool, list | None]:
    """Fallback: check Blockscout for verification and ABI. Returns (is_verified, abi_or_None)."""
    base_url = BLOCKSCOUT_URLS.get(chain)
    if not base_url:
        return False, None
    try:
        r = requests.get(f"{base_url}/api/v2/smart-contracts/{address}", timeout=15)
        if r.status_code == 404:
            return False, None
        r.raise_for_status()
        data = r.json()
        abi  = data.get("abi")
        return True, abi if isinstance(abi, list) else None

    except requests.RequestException:
        return False, None


for p in pools:
    addr     = p["address"]
    chain    = p["chain"]
    chain_id = CHAIN_IDS[chain]
    print(f"  Checking {addr[:10]}...  [{chain}] {p['token0']}/{p['token1']}", end="", flush=True)

    verified, abi = sourcify_check(addr, chain_id)
    source = "Sourcify"

    if not verified:
        verified, abi = blockscout_check(addr, chain)
        source = "Blockscout"

    p["etherscan_verified"] = verified

    if verified and abi:
        abi_path = ABI_DIR / f"{addr}.json"
        abi_path.write_text(json.dumps(abi, indent=2))
        print(f"  ✓ {source}  ({len(abi)} ABI entries → pool_abis/{addr}.json)")
    elif verified:
        print(f"  ✓ {source}  (ABI unavailable)")
    else:
        print(f"  ✗ not verified")

    time.sleep(RATE_LIMIT_DELAY)

## 4. Save Results

In [5]:
df = pd.DataFrame(pools, columns=[
    "address", "chain", "token0", "token1", "fee_tier",
    "tvl_usd", "volume_usd", "etherscan_verified", "llama_id"
])
df.to_parquet(PARQUET_PATH, index=False, engine="pyarrow")

verified_count = df["etherscan_verified"].sum()
abi_files      = list(ABI_DIR.glob("*.json"))

print(f"top_pools.parquet  → {len(df)} rows saved to {PARQUET_PATH}")
print(f"pool_abis/         → {len(abi_files)} ABI files ({verified_count} verified pools)")
print()
df[["chain", "token0", "token1", "fee_tier", "tvl_usd", "etherscan_verified"]]

top_pools.parquet  → 63 rows saved to data\top_pools.parquet
pool_abis/         → 20 ABI files (19 verified pools)



,chain,token0,token1,fee_tier,tvl_usd,etherscan_verified
0,Ethereum,USDC,WETH,500,95980263.0,True
1,Ethereum,WETH,USDT,3000,64839313.0,True
2,Ethereum,WBTC,WETH,500,46502714.0,True
3,Ethereum,WBTC,WETH,3000,44304648.0,True
4,Ethereum,WBTC,CBBTC,100,29150519.0,True
...,...,...,...,...,...,...
58,Base,WETH,TICKER,10000,1143679.0,False
59,Polygon,USDC,USDC,100,5459590.0,False
60,Polygon,WMATIC,COLLECT,10000,2591790.0,False
61,Polygon,WBTC,WETH,500,1779178.0,False
